In [3]:
# generate_exercises.jl

lectures_dir = "../lectures"
output_file = "../exercises/exercises.qmd"

# Ensure output directory exists
mkpath(dirname(output_file))

# Safe top-level JSON package detection
const HAS_JSON = try
    @eval using JSON
    true
catch
    false
end

# Regex matching complete exercise fenced divs: ::: {#exr-label ...} ... :::
exr_div_pattern = r":::\s*\{#(exr-[\w-]+)[^\}]*\}[\s\S]*?:::"

# Helper function to extract clean Markdown text from Jupyter Notebook JSON
function extract_ipynb_markdown(content::String)
    markdown_texts = String[]
    
    if HAS_JSON
        try
            data = JSON.parse(content)
            for cell in get(data, "cells", [])
                if get(cell, "cell_type", "") == "markdown"
                    src = get(cell, "source", "")
                    if src isa Vector
                        push!(markdown_texts, join(src))
                    elseif src isa String
                        push!(markdown_texts, src)
                    end
                end
            end
            return markdown_texts
        catch
        end
    end

    # Base Julia fallback (zero external dependencies)
    chunks = split(content, "\"cell_type\"")
    for chunk in chunks[2:end]
        if startswith(lstrip(chunk), ":") && contains(chunk, "\"markdown\"")
            m = match(r"\"source\"\s*:\s*\[([\s\S]*?)\]", chunk)
            if m !== nothing
                raw_source = m.captures[1]
                lines = String[]
                for str_match in eachmatch(r"\"((?:[^\"\\]|\\.)*)\"", raw_source)
                    push!(lines, Base.unescape_string(str_match.captures[1]))
                end
                push!(markdown_texts, join(lines))
            end
        end
    end
    
    return markdown_texts
end

struct ExerciseItem
    lecture_num::String
    ex_num::Int
    content::String
end

collected_exercises = ExerciseItem[]

if isdir(lectures_dir)
    # Collect matching files
    all_files = String[]
    for (root, _, files) in walkdir(lectures_dir)
        for file in files
            if any(endswith.(file, [".qmd", ".md", ".ipynb"]))
                push!(all_files, joinpath(root, file))
            end
        end
    end

    # Sort files numerically by lecture integer (e.g., L1, L2, L10)
    function parse_lecture_num(path::String)
        m = match(r"L(\d+)"i, basename(path))
        return m !== nothing ? parse(Int, m.captures[1]) : 99999
    end
    sort!(all_files, by=parse_lecture_num)

    for path in all_files
        filename = basename(path)
        
        # Match L followed by digits (e.g. L00, L01, L1)
        m_lec = match(r"L(\d+)"i, filename)
        if m_lec === nothing
            continue
        end

        lec_int = parse(Int, m_lec.captures[1])
        
        # Rule: Ignore Lecture 00
        if lec_int == 0
            continue
        end

        # Rule: Strip leading zeros ("01" -> "1")
        lec_num = string(lec_int)

        file_exercises = String[]

        # Process .qmd and .md files
        if any(endswith.(filename, [".qmd", ".md"]))
            try
                content = read(path, String)
                for m in eachmatch(exr_div_pattern, content)
                    push!(file_exercises, m.match)
                end
            catch e
                println("Error reading $path: $e")
            end
            
        # Process .ipynb notebook files
        elseif endswith(filename, ".ipynb")
            try
                content = read(path, String)
                md_cells = extract_ipynb_markdown(content)
                
                for cell_text in md_cells
                    fenced_matches = [m.match for m in eachmatch(exr_div_pattern, cell_text)]
                    if !isempty(fenced_matches)
                        append!(file_exercises, fenced_matches)
                    elseif contains(cell_text, "{#exr-")
                        push!(file_exercises, cell_text)
                    end
                end
            catch e
                println("Error parsing $path: $e")
            end
        end

        # Format exercise headers using custom .exercise-box CSS class
        ex_counter = 1
        for ex_text in file_exercises
            m_opening = match(r":::\s*\{#(exr-[\w-]+)[^\}]*\}", ex_text)
            formatted_content = ex_text
            
            if m_opening !== nothing
                opening_tag = m_opening.match
                m_id = match(r"exr-[\w-]+", opening_tag)
                
                if m_id !== nothing
                    exr_id = m_id.match
                    
                    # Extract title="..." or title='...' attribute (handles apostrophes & escaped quotes)
                    m_title = match(r"title\s*=\s*(?:\"([^\"]+)\"|'([^']+)')", opening_tag)
                    
                    if m_title !== nothing
                        raw_title = m_title.captures[1] !== nothing ? m_title.captures[1] : m_title.captures[2]
                        title_text = replace(raw_title, "\\'" => "'", "\\\"" => "\"")
                        new_opening = "::: {.exercise-box}\n**@$(exr_id): $(title_text)**\n"
                    else
                        new_opening = "::: {.exercise-box}\n**@$(exr_id)**\n"
                    end
                    
                    formatted_content = replace(ex_text, opening_tag => new_opening)
                end
            end

            push!(collected_exercises, ExerciseItem(lec_num, ex_counter, formatted_content))
            ex_counter += 1
        end
    end
end

# Write formatted Quarto Markdown output grouped by Lecture
open(output_file, "w") do io
    println(io, "# List of exercises\n")
    println(io, "Below is a list of all the exercises across all lectures:\n")
    
    if isempty(collected_exercises)
        println(io, "No exercises found.")
    else
        current_lecture = ""
        for ex in collected_exercises
            # Insert a new H2 section header when transitioning to a new lecture
            if ex.lecture_num != current_lecture
                current_lecture = ex.lecture_num
                println(io, "\n## Lecture $(current_lecture)\n")
            end
            
            println(io, strip(ex.content))
            println(io, "\n")
        end
    end
end

println("✓ Generated $output_file with $(length(collected_exercises)) full exercises.")

✓ Generated ../exercises/exercises.qmd with 19 full exercises.
